# Daphnet FoG Detection — Improved Pipeline
## Part 2: LOSO Evaluation Pipeline

Compares 6 classifiers using Leave-One-Subject-Out cross-validation:
- RandomForest, LogisticRegression, SVM, MLP, AdaBoost, XGBoost

In [ ]:
from __future__ import annotations

import sys, os, time, json, warnings, logging, pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ── Project path ──
PROJECT_ROOT = Path(os.getcwd()).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from utils.pipeline_utils import (
    get_classifiers, get_param_grids, prepare_fold, preprocess_features,
    train_and_evaluate_classifier, build_base_model, aggregate_results,
    print_results_table, print_fusion_results, youden_threshold, compute_metrics,
    HAS_XGB, HAS_SMOTE,
)

try:
    from xgboost import XGBClassifier
except ImportError:
    pass

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("daphnet")

# ── Constants ──
FS = 64
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 20.0, 4
NPERSEG = min(128, WINDOW_SAMPLES)
K_FEATURES = 60
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20
ZERO_FOG_SUBJECTS = {4, 10}

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "daphnet_improved_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

# Load features from notebook 07
features_path = OUTPUT_DIR / "features.pkl"
with open(features_path, "rb") as f:
    features = pickle.load(f)
log.info("Loaded features for %d subjects from %s", len(features), features_path)

In [ ]:
def run_loso_evaluation(features: Dict):
    """Run full LOSO evaluation for all classifiers."""
    classifiers = get_classifiers(SEED)
    param_grids = get_param_grids()
    subjects = sorted(features.keys())

    all_results = {name: [] for name in classifiers}

    for test_sid in tqdm(subjects, desc="LOSO folds"):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)

        # Check if test fold has both classes
        has_both = len(np.unique(y_test)) == 2
        n_fog = int(np.sum(y_test == 1))

        log.info("Fold S%02d: test=%d samples (%d FoG), train=%d samples, both_classes=%s",
                 test_sid, len(y_test), n_fog, len(y_train), has_both)

        # Preprocess
        X_train_p, X_test_p, sel_cols, pipes = preprocess_features(X_train, X_test, y_train, k=K_FEATURES)

        # Train classifiers in parallel
        def _train_clf(clf_name):
            clf = get_classifiers(SEED)[clf_name]
            grid = param_grids[clf_name]
            m = train_and_evaluate_classifier(clf_name, clf, grid, X_train_p, y_train,
                                               X_test_p, y_test, seed=SEED,
                                               n_inner_cv=N_INNER_CV, n_search_iter=N_SEARCH_ITER,
                                               fold_info=f"S{test_sid:02d}")
            m["subject"] = test_sid
            m["has_both_classes"] = has_both
            return clf_name, m

        fold_results = Parallel(n_jobs=-1, verbose=0)(
            delayed(_train_clf)(name) for name in classifiers
        )

        for clf_name, m in fold_results:
            all_results[clf_name].append(m)

    return all_results

In [ ]:
# Step 3: LOSO evaluation
log.info("Running LOSO evaluation with %d classifiers...", len(get_classifiers(SEED)))
all_results = run_loso_evaluation(features)

## Classifier Comparison Results

In [ ]:
# Step 4: Print classifier results
clf_rows = print_results_table(all_results)

In [ ]:
# Save LOSO results for notebook 09
import pickle

loso_results_path = OUTPUT_DIR / "all_results.pkl"
with open(loso_results_path, "wb") as f:
    pickle.dump(all_results, f, protocol=pickle.HIGHEST_PROTOCOL)
log.info("LOSO results saved to %s", loso_results_path)